In [ ]:
"""
Class 31. Introduction to Agent Agents

Installation: 
- Download and Install ollama software.

Ollama Commands:
1. Download a model - ollama pull <MODEL>
2. Start a model and chat with it - ollama run <MODEL>
3. Show all downloaded models - ollama list
4. Show currently running models - ollama ps
5. Stop running a model - ollama stop <MODEL>
6. Delete a model - ollama rm <MODEL>
7. Show model details - ollama show <MODEL>
8. Duplicate a model - ollama cp <MODEL> <NEW_MODEL_NAME>
"""

In [ ]:
import requests
import json
import os

In [ ]:
OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL = "gpt-oss:20b"
MEMORY_FILE = "agent_memory.json"

In [ ]:
""" Memory Management """

def load_memory() -> dict:
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, "r") as f:
            return json.load(f)
    return {"important_senders": [], "important_topics": [], "notes": []}

def save_memory(memory: dict):
    with open(MEMORY_FILE, "w") as f:
        json.dump(memory, f, indent=2)

In [ ]:
""" Tools 
Tool 1: Remember a person important for future emails
Tool 2: Save a short note 
Tool 3: Prepare a draft email  
"""
def tool_save_important_contact(name: str, reason: str) -> str:
    memory = load_memory()
    memory["important_senders"].append({"name": name, "reason": reason})
    save_memory(memory)
    return f"Saved '{name}' as an important contact. Reason: {reason}"

def tool_save_note(note: str) -> str:
    memory = load_memory()
    memory["notes"].append(note)
    save_memory(memory)
    return f"Note saved: {note}"

def tool_draft_reply(tone: str, key_points: str) -> str:
    prompt = (
        f"Write a short, {tone} email reply covering these points:\n"
        f"{key_points}\n\nKeep it under 80 words."
    )
    return call_ollama_simple(prompt)

AVAILABLE_TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "save_important_contact",
            "description": "Remember a sender as important for future reference",
            "parameters": {
                "type": "object",
                "properties": {
                    "name": {"type": "string", "description": "Sender's name"},
                    "reason": {"type": "string", "description": "Why they are important"},
                },
                "required": ["name", "reason"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "save_note",
            "description": "Save a short reminder note about this email for the user",
            "parameters": {
                "type": "object",
                "properties": {
                    "note": {"type": "string", "description": "The note text"},
                },
                "required": ["note"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "draft_reply",
            "description": "Draft a short reply email if a reply is needed",
            "parameters": {
                "type": "object",
                "properties": {
                    "tone": {"type": "string", "description": "e.g. friendly, formal, brief"},
                    "key_points": {"type": "string", "description": "Points to include in the reply"},
                },
                "required": ["tone", "key_points"],
            },
        },
    },
]

In [ ]:
def call_ollama_simple(prompt: str) -> str:
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={"model": MODEL, "prompt": prompt, "stream": False},
    )
    response.raise_for_status()
    return response.json()["response"].strip()

In [ ]:
def run_agent(email_text: str, max_steps: int = 5):
    memory = load_memory()

    system_prompt = f"""You are an email assistant agent.

Your job for each email:
1. Summarize it briefly (2-3 sentences).
2. Decide a category: Urgent, Action Needed, FYI, or Spam.
3. If the sender or topic seems important, call save_important_contact or save_note.
4. If a reply is clearly expected, call draft_reply.
5. When fully done, respond with plain text starting with "FINAL:" followed by
   your summary, category, and any draft reply.

Known important contacts so far: {memory['important_senders']}
Known notes so far: {memory['notes']}

Use tools only when genuinely useful. Don't overuse them."""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"New email:\n\n{email_text}"},
    ]

    print("\n--- AGENT THINKING ---")

    for step in range(max_steps):
        response = requests.post(
            OLLAMA_URL,
            json={
                "model": MODEL,
                "messages": messages,
                "tools": TOOL_DEFINITIONS,
                "stream": False,
            },
        )
        response.raise_for_status()
        msg = response.json()["message"]

        # Case 1: model wants to use a tool
        if msg.get("tool_calls"):
            messages.append(msg)
            for call in msg["tool_calls"]:
                fn_name = call["function"]["name"]
                fn_args = call["function"]["arguments"]

                print(f"[Step {step+1}] Agent is using tool: {fn_name}({fn_args})")

                if fn_name in AVAILABLE_TOOLS:
                    result = AVAILABLE_TOOLS[fn_name](**fn_args)
                else:
                    result = f"Unknown tool: {fn_name}"

                messages.append({
                    "role": "tool",
                    "content": result,
                })

        # Case 2: model gives final answer
        else:
            print(f"[Step {step+1}] Agent finished reasoning.")
            return msg["content"]

In [ ]:
if __name__ == "__main__":
    sample_email = """
    Hi Masum,

    Hope you're doing well. I wanted to follow up on the course proposal you sent last week.
    The team reviewed it and we're generally positive about the structure, but we'd like a few changes:
    1) Add a section on real-world deployment.
    2) Reduce the theory portion in week 2.
    3) Confirm if you can deliver the first session by July 5th.

    Let us know your thoughts by Friday so we can finalize the schedule.

    Best,
    Rafiq
    """

    result = run_agent(sample_email)
    print("\n--- FINAL OUTPUT ---")
    print(result)

    print("\n--- AGENT MEMORY (saved to agent_memory.json) ---")
    print(json.dumps(load_memory(), indent=2))


In [ ]:
""" Task: Simple coding agent.
Given a directory path
1. Create a new folder/file if needed
2. Save the response code in the file

Tools: n8n, langchain, langgraph, crew AI


"""